In [0]:
from pyspark.sql import functions as F
 
CATALOG    = "capstone_five"
LANDING    = f"/Volumes/{CATALOG}/project/landing/vitals/"
CP         = f"/Volumes/{CATALOG}/project/checkpoints/bronze_vitals"          # clean, not nested
SCHEMA_LOC = f"/Volumes/{CATALOG}/project/checkpoints/bronze_vitals_schema"   # clean, not nested
TARGET     = f"{CATALOG}.bronze.vitals"
 
print("checkpoint:", CP)   # double-check before running
 
q = (spark.readStream.format("cloudFiles")
        .option("cloudFiles.format", "json")
        .option("cloudFiles.schemaLocation", SCHEMA_LOC)
        .option("cloudFiles.inferColumnTypes", "true")
        .option("cloudFiles.includeExistingFiles", "true")   # read files that are already there
        .load(LANDING)
        .withColumn("_ingest_ts", F.current_timestamp())
        .withColumn("_source_file", F.col("_metadata.file_path"))
     .writeStream
        .option("checkpointLocation", CP)
        .trigger(availableNow=True)
        .toTable(TARGET))
 
q.awaitTermination()
print("done")

In [0]:
TARGET = "capstone_five.bronze.vitals"
display(spark.sql(f"SELECT count(*) AS total_rows, count(DISTINCT _source_file) AS files FROM {TARGET}"))
display(spark.sql(f"SELECT _source_file, count(*) n FROM {TARGET} GROUP BY 1 HAVING count(*) <> 30"))